In [ ]:
# Import libraries
import os
from pathlib import Path
import requests
import pandas as pd
import time
from dotenv import load_dotenv


# Definir o diretório raiz do repositório
REPO_ROOT = Path('/home/guilherme/Documentos/GitHub/ironstone')
INSTANCES_DIR = REPO_ROOT / 'Instances'

# Definir a versão da instância
INSTANCE_VERSION = 'V12' 

# Definir a hora de partida
DEPART_AT = '2026-11-14T04:00:00'

# Definir a chave da API do TomTom num arquivo .env
load_dotenv(REPO_ROOT / '.env')
API_Key = os.environ.get('TOMTOM_API_KEY', '').strip()
if not API_Key:
    raise ValueError(
        'Defina TOMTOM_API_KEY no .env'
    )

print(f'Instância: {INSTANCE_VERSION}')
print(f'Pasta: {INSTANCES_DIR}')
print(f'Partida: {DEPART_AT}')

In [ ]:
# adaptado para versão json da API TomTom 
def tempo_a_ate_b(lat_inicial, long_inicial, lat_final, long_final, depart_at=None):
    if str(lat_inicial) == str(lat_final) and str(long_inicial) == str(long_final):
        return 0.0

    url = (
        "https://api.tomtom.com/routing/1/calculateRoute/"
        + str(lat_inicial) + "%2C" + str(long_inicial)
        + "%3A" + str(lat_final) + "%2C" + str(long_final)
        + "/json?avoid=unpavedRoads&traffic=true&key=" + API_Key
    )
    if depart_at:
        url += f"&departAt={depart_at}"

    r = requests.get(url, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(
            f"TomTom routing HTTP {r.status_code}: {r.text[:600]}"
        )

    data = r.json()
    routes = data.get("routes") or []
    if not routes:
        raise RuntimeError(
            "Não há rotas disponíveis"
        )

    travel_sec = routes[0]["summary"]["travelTimeInSeconds"]
    return round(travel_sec / 60.0, 1)

print("tempo_a_ate_b carregado")

In [ ]:
# Retrieve the input data for the problem and store them in dataframes.
#df_carros = pd.read_excel(INSTANCES_DIR / f'Instances {INSTANCE_VERSION}_cars.xlsx')
#df_ocorrencias = pd.read_excel(INSTANCES_DIR / f'Instances {INSTANCE_VERSION}_occurrences.xlsx')
#df_atendimento = pd.read_excel(INSTANCES_DIR / f'Instances {INSTANCE_VERSION}_times.xlsx')
#df_disponibilidade = pd.read_excel(INSTANCES_DIR / f'Instances {INSTANCE_VERSION}_availability.xlsx')

df = pd.read_excel(INSTANCES_DIR / f'Instances {INSTANCE_VERSION}.xlsx')

print(f'Dados carregados: {len(df)} pontos')

In [ ]:
# Calculate the travel time between each pair of points. Return the Displacement Matrix.
t0 = time.time()
n = len(df)

#print(f'Gerando matriz {n}x{n} ({n*n} chamadas TomTom)')
matriz = []
for i in range(n):
    lat_inicial = str(df['Lat'][i])
    long_inicial = str(df['Lon'][i])
    
    linha = []
    for j in range(n):
        lat_final = str(df['Lat'][j])
        long_final = str(df['Lon'][j])
        tempo = tempo_a_ate_b(lat_inicial, long_inicial, lat_final, long_final, DEPART_AT)
        linha.append(tempo)
    
    matriz.append(linha)
    print(f'  linha {i + 1}/{n} ok')

#matriz_copy = matriz
tf = time.time()
print('-' * 30)
print(f'Matriz gerada em {tf - t0:.1f}s')

In [ ]:
matrix_number = INSTANCE_VERSION.lstrip('V')
if DEPART_AT:
    hora = DEPART_AT.split('T')[1][:2]
    output_path = INSTANCES_DIR / f'matrix{matrix_number}_{hora}h.txt'

else:
    output_path = INSTANCES_DIR / f'matrix{matrix_number}.txt'
output_path.write_text(str(matriz), encoding='utf-8')
print(f'Matriz {n}x{n} salva em {output_path}')